**Introduction**
- This notebook is workplace for experimenting and sampling synthetic data by FairFinGAN-SP model from the research work: "FairFinGAN: Fairness-aware Synthetic Financial Data (2025)".
- This notebook consists of 4 parts: *Utils:* some preprocessing transform adapting to the original tabular datasets; *GANModules*: including architectures of GAN, proposed components MLP; the *Training* and *Generating* processes.

# 1.Utils


In [ ]:
import torch
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from collections import OrderedDict
from typing import Tuple, List, Dict, Union, Optional

from sklearn.preprocessing import OneHotEncoder, QuantileTransformer
from sklearn.model_selection import train_test_split



def get_cont_cat_transform(df: pd.DataFrame) -> Tuple[
    OneHotEncoder,
    QuantileTransformer,
    OrderedDict,
    List[str],
    np.ndarray,
    np.ndarray
]:
    df_int = df.select_dtypes(['float', 'integer']).values
    continuous_columns_list = list(df.select_dtypes(['float', 'integer']).columns)
    scaler = QuantileTransformer(n_quantiles=2000, output_distribution='uniform')
    df_int = scaler.fit_transform(df_int)

    df_cat = df.select_dtypes('object')
    df_cat_names = list(df.select_dtypes('object').columns)
    numerical_array = df_int
    ohe = OneHotEncoder()
    ohe_array = ohe.fit_transform(df_cat)

    cat_lens = [i.shape[0] for i in ohe.categories_]
    discrete_columns_ordereddict = OrderedDict(zip(df_cat_names, cat_lens))

    return ohe, scaler, discrete_columns_ordereddict, continuous_columns_list, numerical_array, ohe_array



#def get_ohe_data_fair(df: pd.DataFrame, S, Y, S_under, Y_desire):
def get_ohe_data_fair(
    df: pd.DataFrame,
    S: str,
    Y: str,
    S_under: str,
    Y_desire: str
) -> Tuple[
    OneHotEncoder,
    QuantileTransformer,
    OrderedDict,
    List[str],
    np.ndarray,
    int,
    int,
    int,
    int,
    int,
    int
]:

    ohe, scaler, discrete_columns_ordereddict, continuous_columns_list, numerical_array, ohe_array = get_cont_cat_transform(df)

    S_start_index = len(continuous_columns_list) + sum(
        list(discrete_columns_ordereddict.values())[:list(discrete_columns_ordereddict.keys()).index(S)])
    Y_start_index = len(continuous_columns_list) + sum(
        list(discrete_columns_ordereddict.values())[:list(discrete_columns_ordereddict.keys()).index(Y)])

    if ohe.categories_[list(discrete_columns_ordereddict.keys()).index(S)][0] == S_under:
        underpriv_index = 0
        priv_index = 1
    else:
        underpriv_index = 1
        priv_index = 0
    if ohe.categories_[list(discrete_columns_ordereddict.keys()).index(Y)][0] == Y_desire:
        desire_index = 0
        undesire_index = 1
    else:
        desire_index = 1
        undesire_index = 0

    final_array = np.hstack((numerical_array, ohe_array.toarray()))
    return ohe, scaler, discrete_columns_ordereddict, continuous_columns_list, final_array, S_start_index, Y_start_index, underpriv_index, priv_index, undesire_index, desire_index




#def get_ohe_data_nofair(df: pd.DataFrame):
def get_ohe_data_nofair(df: pd.DataFrame) -> Tuple[
    OneHotEncoder,
    QuantileTransformer,
    OrderedDict,
    List[str],
    np.ndarray
]:
    ohe, scaler, discrete_columns_ordereddict, continuous_columns_list, numerical_array, ohe_array = get_cont_cat_transform(df)


    final_array = np.hstack((numerical_array, ohe_array.toarray()))
    return ohe, scaler, discrete_columns_ordereddict, continuous_columns_list, final_array


#def get_original_data(df_transformed, df_orig, ohe, scaler):
def get_original_data(
    df_transformed: np.ndarray,
    df_orig: pd.DataFrame,
    ohe: OneHotEncoder,
    scaler: QuantileTransformer
) -> pd.DataFrame:
    df_ohe_int = df_transformed[:, :df_orig.select_dtypes(['float', 'integer']).shape[1]]
    df_ohe_int = scaler.inverse_transform(df_ohe_int)
    df_ohe_cats = df_transformed[:, df_orig.select_dtypes(['float', 'integer']).shape[1]:]
    df_ohe_cats = ohe.inverse_transform(df_ohe_cats)
    df_int = pd.DataFrame(df_ohe_int, columns=df_orig.select_dtypes(['float', 'integer']).columns)
    df_cat = pd.DataFrame(df_ohe_cats, columns=df_orig.select_dtypes('object').columns)
    return pd.concat([df_int, df_cat], axis=1)


#def prepare_data_fair(df, batch_size, S, Y, S_under, Y_desire):
def prepare_data_fair(
    df: pd.DataFrame,
    batch_size: int,
    S: str,
    Y: str,
    S_under: str,
    Y_desire: str
) -> Tuple[
    OneHotEncoder,
    QuantileTransformer,
    int,
    List[str],
    List[str],
    DataLoader,
    np.ndarray,
    np.ndarray,
    int,
    int,
    int,
    int,
    int,
    int
]:
    ohe, scaler, discrete_columns, continuous_columns, df_transformed, S_start_index, Y_start_index, underpriv_index, priv_index, undesire_index, desire_index = get_ohe_data_fair(df, S, Y, S_under, Y_desire)
    input_dim = df_transformed.shape[1]
    X_train, X_test = train_test_split(df_transformed,test_size=0.1, shuffle=True)
    data_train = X_train.copy()
    data_test = X_test.copy()


    data = torch.from_numpy(data_train).float()


    train_ds = TensorDataset(data)
    train_dl = DataLoader(train_ds, batch_size = batch_size, drop_last=True)
    return ohe, scaler, input_dim, discrete_columns, continuous_columns ,train_dl, data_train, data_test, S_start_index, Y_start_index, underpriv_index, priv_index, undesire_index, desire_index

#def prepare_data_nofair(df, batch_size):
def prepare_data_nofair(df: pd.DataFrame, batch_size: int) -> Tuple[
    OneHotEncoder,
    QuantileTransformer,
    int,
    List[str],
    List[str],
    DataLoader,
    np.ndarray,
    np.ndarray
]:

    ohe, scaler, discrete_columns, continuous_columns, df_transformed = get_ohe_data_nofair(df)


    input_dim = df_transformed.shape[1]

    X_train, X_test = train_test_split(df_transformed,test_size=0.1, shuffle=True) #random_state=10)

    data_train = X_train.copy()
    data_test = X_test.copy()

    data = torch.from_numpy(data_train).float()

    train_ds = TensorDataset(data)
    train_dl = DataLoader(train_ds, batch_size = batch_size, drop_last=True)
    return ohe, scaler, input_dim, discrete_columns, continuous_columns, train_dl, data_train, data_test

# 2. GANModules


In [ ]:
import torch
import torch.nn.functional as f
from torch import nn

class Generator(nn.Module):
    def __init__(self, input_dim, continuous_columns, discrete_columns):
        super(Generator, self).__init__()
        self._input_dim = input_dim
        self._discrete_columns = discrete_columns
        self._num_continuous_columns = len(continuous_columns)

        self.lin1 = nn.Linear(self._input_dim, self._input_dim)
        self.lin_numerical = nn.Linear(self._input_dim, self._num_continuous_columns)

        self.lin_cat = nn.ModuleDict()
        for key, value in self._discrete_columns.items():
            self.lin_cat[key] = nn.Linear(self._input_dim, value)

    def forward(self, x):
        x = torch.relu(self.lin1(x))
        
        x_numerical = f.relu(self.lin_numerical(x))
        x_cat = []
        for key in self.lin_cat:
            x_cat.append(f.gumbel_softmax(self.lin_cat[key](x), tau=0.2))
        x_final = torch.cat((x_numerical, *x_cat), 1)
        return x_final


class Critic(nn.Module):
    def __init__(self, input_dim):
        super(Critic, self).__init__()
        self._input_dim = input_dim
        
        self.dense1 = nn.Linear(self._input_dim, self._input_dim)
        self.dense2 = nn.Linear(self._input_dim, self._input_dim)
        

    def forward(self, x):
        x = f.leaky_relu(self.dense1(x))
        
        x = f.leaky_relu(self.dense2(x))
        
        return x


class FairLossFunc(nn.Module):
    def __init__(self, S_start_index, Y_start_index, underpriv_index, priv_index, undesire_index, desire_index, classifier):
        super(FairLossFunc, self).__init__()
        self._S_start_index = S_start_index
        self._Y_start_index = Y_start_index
        self._underpriv_index = underpriv_index
        self._priv_index = priv_index
        self._undesire_index = undesire_index
        self._desire_index = desire_index
        self._classifier = classifier                         # proposed pretrained-classifier

    def forward(self, x, crit_fake_pred, lamda):
        G = x[:, self._S_start_index:self._S_start_index + 2]
        
#*****************************************************************************************************************************
        # get the x_h = x without label Y 
        x_h = torch.cat((
            x[:, :self._Y_start_index],
            x[:, self._Y_start_index + 2:]
        ), dim=1)

        logits = self._classifier(x_h)
        I = torch.nn.functional.gumbel_softmax(logits, tau=0.2, hard=False)

#*****************************************************************************************************************************


        disp = -1.0 * lamda * (torch.mean(G[:, self._underpriv_index] * I[:, self._desire_index]) / (
            x[:, self._S_start_index + self._underpriv_index].sum()) - torch.mean(
            G[:, self._priv_index] * I[:, self._desire_index]) / (
                                   x[:, self._S_start_index + self._priv_index].sum())) - 1.0 * torch.mean(
            crit_fake_pred)
        # print(disp)

#*****************************************************************************************************************************

        return disp



def get_gradient(crit, real, fake, epsilon):
    mixed_data = real * epsilon + fake * (1 - epsilon)

    mixed_scores = crit(mixed_data)

    gradient = torch.autograd.grad(
        inputs=mixed_data,
        outputs=mixed_scores,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True,
        retain_graph=True,
    )[0]
    return gradient


def gradient_penalty(gradient):
    gradient = gradient.view(len(gradient), -1)
    gradient_norm = gradient.norm(2, dim=1)

    penalty = torch.mean((gradient_norm - 1) ** 2)
    return penalty


def get_gen_loss(crit_fake_pred):
    gen_loss = -1. * torch.mean(crit_fake_pred)

    return gen_loss


def get_crit_loss(crit_fake_pred, crit_real_pred, gp, c_lambda):
    crit_loss = torch.mean(crit_fake_pred) - torch.mean(crit_real_pred) + c_lambda * gp

    return crit_loss

## 2.a. Pretrained Classifier (MLP)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ClassifierMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64)):
        super(ClassifierMLP, self).__init__()
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden_dims[0]),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dims[0], hidden_dims[1]),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dims[1], 2),
        )

    def forward(self, x):
        return self.mlp(x)

    def train_model(self, train_loader, num_epochs=50, lr=0.001):
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)

        for epoch in range(num_epochs):
            self.train()  # switch to training mode
            total_loss = 0.0

            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                outputs = self(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

    def evaluate_model(self, test_loader):
        self.eval()  # switch to evaluation mode
        correct = 0
        total = 0

        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = self(batch_X)
                _, predicted = torch.max(outputs, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()

        accuracy = correct / total
        print(f"Test Accuracy: {accuracy * 100:.2f}%")
        return accuracy

# 3. Training

In [ ]:
import torch
import torch.nn.functional as f
from torch import nn
import pandas as pd
import numpy as np
from collections import OrderedDict

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import train_test_split
import argparse
from tqdm.auto import tqdm




display_step = 50


class TFG:

    def __init__(self, df: pd.DataFrame, epochs: int, batch_size: int, device: str = None, fairness_config: dict = None) -> None:
        self.df = df
        self.epochs = epochs
        self.batch_size = batch_size
        self.device = "cpu" if device is None else device

        # Fairness configuration
        self.fair_epochs = fairness_config.get('fair_epochs', 0) if fairness_config else 0

        if fairness_config and self.fair_epochs > 0:
            self.lamda = fairness_config.get('lamda')
            self.S = fairness_config.get('S')
            self.Y = fairness_config.get('Y')
            self.S_under = fairness_config.get('S_under')
            self.Y_desire = fairness_config.get('Y_desire')

            # Perform type checking
            if not isinstance(self.lamda, float):
                raise TypeError("When fair_epochs is > 0, 'lamda' must be a float.")
            if not isinstance(self.S, str):
                raise TypeError("When fair_epochs is > 0, 'S' must be a string.")
            if not isinstance(self.Y, str):
                raise TypeError("When fair_epochs is > 0, 'Y' must be a string.")
            if not isinstance(self.S_under, str):
                raise TypeError("When fair_epochs is > 0, 'S_under' must be a string.")
            if not isinstance(self.Y_desire, str):
                raise TypeError("When fair_epochs is > 0, 'Y_desire' must be a string.")
        else:
            self.lamda = None
            self.S = None
            self.Y = None
            self.S_under = None
            self.Y_desire = None


    def prepare_data(self) -> None:
        if self.fair_epochs > 0:
            self.ohe, self.scaler, self.input_dim, self.discrete_columns, self.continuous_columns, self.train_dl, self.data_train, self.data_test, self.S_start_index, self.Y_start_index, self.underpriv_index, self.priv_index, self.undesire_index, self.desire_index = prepare_data_fair(self.df, self.batch_size, self.S, self.Y, self.S_under, self.Y_desire)

        else:
            self.ohe, self.scaler, self.input_dim, self.discrete_columns, self.continuous_columns, self.train_dl, self.data_train, self.data_test = prepare_data_nofair(self.df, self.batch_size)


    def train(self) -> None:

        self.prepare_data()

#**************************************************************************************************************************************
        self.simpleClassifier = ClassifierMLP(input_dim=self.input_dim - 2)

        X_real = pd.DataFrame(self.data_train).iloc[:, :-2].values
        y_real = pd.DataFrame(self.data_train).iloc[:, -2:].values
        y_real = np.argmax(y_real, axis=1)

        Xtr_real, Xts_real, Ytr_real, Yts_real = train_test_split(X_real, y_real, test_size=0.2, random_state=42)

        print(Xtr_real.shape)

        Xtr_tensor = torch.tensor(Xtr_real, dtype=torch.float32)
        Ytr_tensor = torch.tensor(Ytr_real, dtype=torch.long)

        Xts_tensor = torch.tensor(Xts_real, dtype=torch.float32)
        Yts_tensor = torch.tensor(Yts_real, dtype=torch.long)

        # DataLoader
        train_dataset = TensorDataset(Xtr_tensor, Ytr_tensor)
        test_dataset = TensorDataset(Xts_tensor, Yts_tensor)

        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
        test_loader = DataLoader(test_dataset, batch_size=32)

        self.simpleClassifier.train_model(train_loader=train_loader)

        self.simpleClassifier.evaluate_model(test_loader=test_loader)

#**************************************************************************************************************************************
        self.generator = Generator(self.input_dim, self.continuous_columns, self.discrete_columns).to(self.device)

        self.critic = Critic(self.input_dim).to(self.device)

        if self.fair_epochs > 0:
            self.second_critic = FairLossFunc(self.S_start_index, self.Y_start_index, self.underpriv_index, self.priv_index, self.undesire_index, self.desire_index, self.simpleClassifier).to(self.device)

        self.gen_optimizer = torch.optim.Adam(self.generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
        self.gen_optimizer_fair = torch.optim.Adam(self.generator.parameters(), lr=0.0001, betas=(0.5, 0.999))
        self.crit_optimizer = torch.optim.Adam(self.critic.parameters(), lr=0.0002, betas=(0.5, 0.999))


        critic_losses = []
        cur_step = 0
        with tqdm(total=self.epochs, desc="Training Progress", ncols=100) as pbar:
            for i in range(self.epochs):
                print("epoch {}".format(i + 1))
                ############################
                if i + 1 <= (self.epochs - self.fair_epochs):
                    pbar.set_postfix_str("Training for accuracy")
                    #print("training for accuracy")
                if i + 1 > (self.epochs - self.fair_epochs):
                    pbar.set_postfix_str("Training for fairness")
                    #print("training for fairness")
                for data in self.train_dl:
                    data[0] = data[0].to(self.device)
                    crit_repeat = 4
                    mean_iteration_critic_loss = 0
                    for k in range(crit_repeat):
                        # training the critic
                        self.crit_optimizer.zero_grad()
                        fake_noise = torch.randn(size=(self.batch_size, self.input_dim), device=self.device).float()
                        fake = self.generator(fake_noise)

                        crit_fake_pred = self.critic(fake.detach())
                        crit_real_pred = self.critic(data[0])

                        epsilon = torch.rand(self.batch_size, self.input_dim, device=self.device, requires_grad=True)
                        gradient = get_gradient(self.critic, data[0], fake.detach(), epsilon)
                        gp = gradient_penalty(gradient)

                        crit_loss = get_crit_loss(crit_fake_pred, crit_real_pred, gp, c_lambda=10)

                        mean_iteration_critic_loss += crit_loss.item() / crit_repeat
                        crit_loss.backward(retain_graph=True)
                        self.crit_optimizer.step()
                    #############################
                    if cur_step > 50:
                        critic_losses += [mean_iteration_critic_loss]

                    #############################
                    if i + 1 <= (self.epochs - self.fair_epochs):
                        # training the generator for accuracy
                        self.gen_optimizer.zero_grad()
                        fake_noise_2 = torch.randn(size=(self.batch_size, self.input_dim), device=self.device).float()
                        fake_2 = self.generator(fake_noise_2)
                        crit_fake_pred = self.critic(fake_2)

                        gen_loss = get_gen_loss(crit_fake_pred)
                        gen_loss.backward()

                        # Update the weights
                        self.gen_optimizer.step()

                    ###############################
                    if i + 1 > (self.epochs - self.fair_epochs):
                        # training the generator for fairness
                        self.gen_optimizer_fair.zero_grad()
                        fake_noise_2 = torch.randn(size=(self.batch_size, self.input_dim), device=self.device).float()
                        fake_2 = self.generator(fake_noise_2)

                        crit_fake_pred = self.critic(fake_2)

                        gen_fair_loss = self.second_critic(fake_2, crit_fake_pred, self.lamda)

                        gen_fair_loss.backward()
                        self.gen_optimizer_fair.step()


                    cur_step += 1

                pbar.update(1)


    def generate_fake_df(self, num_rows: int) -> pd.DataFrame:
        with torch.no_grad():
            fake_numpy_array = self.generator(torch.randn(size=(num_rows, self.input_dim), device=self.device)).cpu().detach().numpy()

            fake_df = get_original_data(fake_numpy_array, self.df, self.ohe, self.scaler)

            fake_df = fake_df[self.df.columns]
            return fake_df

# 4. Generating

In [ ]:
df = pd.read_csv("adult.csv")
num = len(df)
print(num)
df['class-label'] = df['class-label'].astype(str)
df['gender'] = df['gender'].astype(str)
df['age'] = df['age'].astype(str)
df['race'] = df['race'].astype(str)

fairness_config = {
    'fair_epochs': 50,
    'lamda': 0.5,
    'S': 'gender',
    'Y': 'class-label',
    'S_under': '0',
    'Y_desire': '1'
}

tfg = TFG(df, epochs=200, batch_size=256, device="cuda:0", fairness_config=fairness_config)

tfg.train()

synthetic_dataset = tfg.generate_fake_df(num)

synthetic_dataset.to_csv('adult_generation_gender.csv', index=False)